In [1]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, AutoTokenizer
from qwen_vl_utils import process_vision_info
import os

/home/jack/anaconda3/envs/qwenvl-new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
checkpoint_path = "/data3/qwen-weights/mmedagent2/models--ZihaoLin--mmedagent2/snapshots/1ac904867da02e4f2704778b4d58c18f8dfa0144"
processor_dir = "/data3/qwen-weights/mmedagent2/models--ZihaoLin--mmedagent2/snapshots/1ac904867da02e4f2704778b4d58c18f8dfa0144"
base_model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

print("Loading fine-tuned model...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    checkpoint_path,
    torch_dtype="auto",
    device_map="auto"
)

print("Loading processor...")
processor = AutoProcessor.from_pretrained(processor_dir)

if not hasattr(processor, 'chat_template') or processor.chat_template is None:
    print("Chat template missing, loading from base model...")
    base_processor = AutoProcessor.from_pretrained(base_model_name)
    processor.chat_template = base_processor.chat_template

print("Loading tokenizer...")
try:
    tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
except:
    tokenizer = AutoTokenizer.from_pretrained(processor_dir)

print("✓ All components loaded successfully!")

Loading fine-tuned model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.03s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


Loading processor...
Chat template missing, loading from base model...
Loading tokenizer...
✓ All components loaded successfully!


In [ ]:
# def chat_with_model(user_input, image_path=None, conversation_history=None):
#     if conversation_history is None:
#         conversation_history = []

#     if image_path and os.path.exists(image_path):
#         user_message = {
#             "role": "human",
#             "content": [
#                 {"type": "image", "image": image_path},
#                 {"type": "text", "text": user_input}
#             ]
#         }
#     else:
#         user_message = {
#             "role": "human",
#             "content": [{"type": "text", "text": user_input}]
#         }
    
#     conversation_history.append(user_message)
    
#     text = processor.apply_chat_template(
#         conversation_history, 
#         tokenize=False, 
#         add_generation_prompt=True
#     )
    
#     image_inputs, video_inputs = process_vision_info(conversation_history)
    
#     inputs = processor(
#         text=[text],
#         images=image_inputs,
#         videos=video_inputs,
#         padding=True,
#         return_tensors="pt"
#     ).to(model.device)
    
#     with torch.no_grad():
#         generated_ids = model.generate(
#             **inputs,
#             max_new_tokens=512,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.8
#         )
    
#     response_ids = [
#         out_ids[len(in_ids):] for in_ids, out_ids in 
#         zip(inputs.input_ids, generated_ids)
#     ]

#     response = processor.batch_decode(
#         response_ids, 
#         skip_special_tokens=True, 
#         clean_up_tokenization_spaces=False
#     )[0]
    
#     assistant_message = {
#         "role": "gpt",
#         "content": [{"type": "text", "text": response}]
#     }
#     conversation_history.append(assistant_message)
    
#     return response, conversation_history

# print("Chat function defined!")

Chat function defined!


In [3]:
def chat_with_model(user_input, image_paths=None, conversation_history=None):
    if conversation_history is None:
        conversation_history = []

    content = []

    if image_paths:
        if isinstance(image_paths, str):
            image_paths = [image_paths]
        
        for image_path in image_paths:
            if os.path.exists(image_path):
                content.append({"type": "image", "image": image_path})
            else:
                print(f"Warning: Image not found: {image_path}")
    
    content.append({"type": "text", "text": user_input})
    
    user_message = {
        "role": "human",
        "content": content
    }
    
    conversation_history.append(user_message)
    
    text = processor.apply_chat_template(
        conversation_history, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    image_inputs, video_inputs = process_vision_info(conversation_history)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.8
        )
    
    response_ids = [
        out_ids[len(in_ids):] for in_ids, out_ids in 
        zip(inputs.input_ids, generated_ids)
    ]

    response = processor.batch_decode(
        response_ids, 
        skip_special_tokens=True, 
        clean_up_tokenization_spaces=False
    )[0]
    
    assistant_message = {
        "role": "gpt",
        "content": [{"type": "text", "text": response}]
    }
    conversation_history.append(assistant_message)
    
    return response, conversation_history

print("Chat function defined!")

Chat function defined!


In [4]:
def print_conversation_history(history):
    """Print the conversation history in a readable format"""
    print("=== Conversation History ===")
    for i, message in enumerate(history):
        role = message["role"].upper()
        content = message["content"]
        print(f"{i+1}. {role}:")
        for item in content:
            if item["type"] == "text":
                print(f"   {item['text']}")
            elif item["type"] == "image":
                print(f"   [Image: {item['image']}]")
        print()

def reset_conversation():
    """Return empty conversation history"""
    return []

In [19]:
import os, re, json, math, time
from typing import List, Dict, Any, Tuple, Optional
from collections import defaultdict, Counter

# ========= config =========
TEST_JSONL = "/home/jack/Projects/yixin-llm/yixin-llm-data/multi_round/MMedAgent-V2/test/test.jsonl"
BASE_DIR = "/home/jack/Projects/yixin-llm/yixin-llm-data/multi_round"
OUT_DIR    = "output_qwen"

os.makedirs(OUT_DIR, exist_ok=True)

In [14]:
API_NAME_KEYS = ["API_name", "api_name", "API", "api", "name"]
API_PARAMS_KEYS = ["API_params", "api_params", "params", "arguments"]

TOOL_OUT_RE = re.compile(r"([A-Za-z0-9_\-]+)\s*output\s*:\s*(.*)", re.IGNORECASE | re.DOTALL)

def norm(s: str) -> str:
    import re
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

def try_json_loads(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_json_blocks(text: str) -> List[str]:
    """尽量从模型输出中抓取 JSON/代码块里的 JSON。"""
    blocks = []
    # ```json ... ```
    for m in re.finditer(r"```(?:json)?\s*(\{[\s\S]*?\})\s*```", text, flags=re.IGNORECASE):
        blocks.append(m.group(1))
    # Action: { ... }
    for m in re.finditer(r"Action\s*:\s*(\{[\s\S]*?\})", text, flags=re.IGNORECASE):
        blocks.append(m.group(1))
    # 裸 JSON（保守地再抓一遍最长的花括号块）
    for m in re.finditer(r"(\{[\s\S]*?\})", text):
        blocks.append(m.group(1))
    # 去重
    out, seen = [], set()
    for b in blocks:
        if b not in seen:
            out.append(b); seen.add(b)
    return out

def parse_actions_from_text(text: str) -> List[Dict[str, Any]]:
    """从模型输出里解析工具调用列表：[{api, params}]"""
    actions = []

    # 1) 优先找 JSON 块
    for jb in extract_json_blocks(text):
        obj = try_json_loads(jb)
        if isinstance(obj, dict):
            # 可能是单个 action
            api, params = None, {}
            for k in API_NAME_KEYS:
                if k in obj:
                    api = obj[k]
                    break
            for k in API_PARAMS_KEYS:
                if k in obj and isinstance(obj[k], dict):
                    params = obj[k]
                    break
            if api:
                actions.append({"api": str(api), "params": params})
        elif isinstance(obj, list):
            # 可能是动作列表
            for it in obj:
                if isinstance(it, dict):
                    api, params = None, {}
                    for k in API_NAME_KEYS:
                        if k in it:
                            api = it[k]; break
                    for k in API_PARAMS_KEYS:
                        if k in it and isinstance(it[k], dict):
                            params = it[k]; break
                    if api:
                        actions.append({"api": str(api), "params": params})

    # 2) 兜底：正则直接从自然语言中抓 “调用 XXX …” 之类的
    if not actions:
        for m in re.finditer(r"(?:call|use|invoke)\s+([A-Za-z0-9_\-]+)", text, flags=re.IGNORECASE):
            actions.append({"api": m.group(1), "params": {}})

    # 3) 归一化名称
    for a in actions:
        a["api"] = a["api"].strip()
    # 去重（保留顺序）
    seen = set(); uniq = []
    for a in actions:
        key = (a["api"], json.dumps(a.get("params", {}), sort_keys=True))
        if key not in seen:
            uniq.append(a); seen.add(key)
    return uniq

def extract_ground_truth_tools_from_gpt_annotations(conversations: List[Dict[str,Any]]) -> List[str]:
    """
    从测试集里标注的 gpt turn 的 actions 字段抽取工具序列，
    这相当于“期望调用”的工具（planner 的意图）。
    """
    gt_seq = []
    for turn in conversations:
        if turn.get("from") == "gpt":
            acts = turn.get("actions") or []
            for a in acts:
                api = a.get("API_name") or a.get("api") or a.get("name")
                if api:
                    gt_seq.append(api)
    return gt_seq

def extract_required_tools_from_human_outputs(conversations: List[Dict[str,Any]]) -> List[str]:
    """
    从 human 的 “X output: ...” 注入中抽取实际出现过的工具，
    这相当于“本样本确实需要/使用过”的工具集合。
    """
    tools = []
    for turn in conversations:
        if turn.get("from") != "human":
            continue
        val = turn.get("value","")
        m = TOOL_OUT_RE.search(val)
        if m:
            tools.append(m.group(1).strip())
    # 去重保序
    seen=set(); out=[]
    for t in tools:
        if t not in seen:
            out.append(t); seen.add(t)
    return out

def extract_depends_on(conversations: List[Dict[str, Any]]) -> List[str]:
    deps = []
    for turn in conversations:
        if "depends_on" in turn:
            for d in (turn.get("depends_on") or []):
                deps.append(d)
    # 去重保序
    seen=set(); out=[]
    for d in deps:
        if d not in seen:
            out.append(d); seen.add(d)
    return out

In [24]:
def set_prf1(pred: List[str], gold: List[str]) -> Tuple[float,float,float]:
    P, G = set(pred), set(gold)
    tp = len(P & G)
    prec = tp / (len(P) or 1)
    rec  = tp / (len(G) or 1)
    f1 = 0.0 if (prec+rec)==0 else 2*prec*rec/(prec+rec)
    return round(prec,4), round(rec,4), round(f1,4)

def check_order_is_subsequence(pred_seq: List[str], ref_seq: List[str]) -> int:
    """检查 ref_seq 是否为 pred_seq 的子序列（允许中间插入其它工具）。"""
    if not ref_seq:
        return 1
    j = 0
    for x in pred_seq:
        if x == ref_seq[j]:
            j += 1
            if j == len(ref_seq):
                return 1
    return 0

def evaluate_one_conversation(sample: Dict[str,Any]) -> Dict[str,Any]:
    """
    驱动你的 chat_with_model，逐轮把 human 的内容喂给模型，
    采集模型每轮输出并解析其中的 tool actions。
    """
    conv = sample.get("conversations", [])
    history = []   # 传给你的 chat_with_model 的对话历史（注意：你的函数里 role 用 "human"/"gpt"）
    inferred_actions = []  # 按轮记录模型宣称的工具调用（解析出的顺序）

    # 逐轮把 human 的消息喂给模型
    for turn in conv:
        if turn.get("from") != "human":
            continue
        val = turn.get("value","")
        # 处理 <image>：你的 chat_with_model 支持传 list[str] 做 image_paths
        # 这里从 sample["image"] 字段里取（兼容 str/list）
        relative_paths = sample.get("image")
        if isinstance(relative_paths, str):
            # 单张图
            image_paths = [os.path.abspath(os.path.join(BASE_DIR, relative_paths))]
        else:
            # 多张图
            image_paths = [os.path.abspath(os.path.join(BASE_DIR, p)) for p in relative_paths]
        # 仅在该 human 消息包含 <image> 或明显在讨论影像时携带图像
        use_images = None
        if isinstance(image_paths, list):
            use_images = image_paths
        elif isinstance(image_paths, str):
            use_images = [image_paths]
        else:
            use_images = None

        # 调用你的模型
        try:
            resp, history = chat_with_model(
                user_input=val,
                image_paths=use_images if ("<image>" in val or any(k in norm(val) for k in ["image","x-ray","ct","mri","oct"])) else None,
                conversation_history=history
            )
        except Exception as e:
            resp = f"[ERROR during generation: {e}]"
            # 把错误也写入历史，避免后续轮次崩
            history = history or []
            history.append({"role": "gpt", "content":[{"type":"text","text":resp}]})

        # 解析这一轮模型输出中的工具调用
        acts = parse_actions_from_text(resp)
        for a in acts:
            inferred_actions.append(a["api"])

    # ========== 计算指标 ==========
    # 1) 基于 human 注入输出的 “必需工具集合”
    required_tools = extract_required_tools_from_human_outputs(conv)   # 集合指标
    # 2) 基于 gpt 标注 actions 的 “期望调用序列”（如果你的测试集里存在）
    gt_seq = extract_ground_truth_tools_from_gpt_annotations(conv)     # 序列指标
    # 3) 依赖顺序（depends_on），例如 ["IterNet","LLaVA"]
    depends_on = extract_depends_on(conv)                              # 序列指标

    # 集合层面 PRF1（用 required_tools 做 gold）
    prec, rec, f1 = set_prf1(inferred_actions, required_tools)

    # 序列层面（如果 gt_seq 存在就评它；否则退化为 depends_on）
    order_ok_by_gt   = check_order_is_subsequence(inferred_actions, gt_seq) if gt_seq else None
    order_ok_by_deps = check_order_is_subsequence(inferred_actions, depends_on) if depends_on else None

    # 首次工具时机：第一条工具调用之前是否已经出现 human 的 “X output”？
    #   理想：模型先宣告调用，再由 human 注入输出 → 这里给一个粗略的合规性标记
    first_model_tool_idx = None if not inferred_actions else next((i for i, t in enumerate(inferred_actions) if t), None)
    saw_human_tool_output = any(TOOL_OUT_RE.search((t.get("value") or "")) for t in conv)
    # 我们把“模型先声明工具，再收到输出”的理想流程定义为 1 分
    timing_ok = 1 if (first_model_tool_idx is not None and saw_human_tool_output) else 0

    return {
        "id": sample.get("id") or (sample.get("image") if isinstance(sample.get("image"), str) else None),
        "inferred_tools_seq": inferred_actions,
        "required_tools_set": required_tools,
        "gt_tools_seq": gt_seq or None,
        "depends_on": depends_on or None,
        "tool_called_set_precision": prec,
        "tool_called_set_recall":    rec,
        "tool_called_set_f1":        f1,
        "order_ok_by_gt":            order_ok_by_gt,
        "order_ok_by_depends_on":    order_ok_by_deps,
        "timing_ok":                 timing_ok,
    }

In [25]:
per_sample_path = os.path.join(OUT_DIR, "tool_use_per_sample.jsonl")
summary_path    = os.path.join(OUT_DIR, "tool_use_summary.json")

results = []
with open(TEST_JSONL, "r", encoding="utf-8") as fin, open(per_sample_path, "w", encoding="utf-8") as fout:
    for li, line in enumerate(fin, 1):
        line = line.strip()
        if not line: 
            continue
        sample = json.loads(line)
        if "id" not in sample:
            sid = sample.get("image")
            if isinstance(sid, list): sid = ",".join(sid)
            sample["id"] = sid or f"sample_{li}"
        r = evaluate_one_conversation(sample)
        results.append(r)
        fout.write(json.dumps(r, ensure_ascii=False) + "\n")

# 汇总
agg = defaultdict(list)
for r in results:
    for k in ["tool_called_set_precision","tool_called_set_recall","tool_called_set_f1","timing_ok"]:
        if k in r and r[k] is not None:
            agg[k].append(float(r[k]))
    for k in ["order_ok_by_gt","order_ok_by_depends_on"]:
        if r.get(k) is not None:
            agg[k].append(float(r[k]))

summary = {k: round(sum(v)/len(v), 4) if v else None for k, v in agg.items()}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("== Done ==")
print("Per-sample metrics:", per_sample_path)
print("Summary metrics   :", summary_path)
print(json.dumps(summary, indent=2, ensure_ascii=False))

== Done ==
Per-sample metrics: output_qwen/tool_use_per_sample.jsonl
Summary metrics   : output_qwen/tool_use_summary.json
{
  "tool_called_set_precision": 0.3891,
  "tool_called_set_recall": 0.4542,
  "tool_called_set_f1": 0.4137,
  "timing_ok": 0.9533,
  "order_ok_by_gt": 0.1933
}


In [5]:
response, history = chat_with_model("Hello! How are you today?")
print(f"Human: Hello! How are you today?")
print(f"GPT: {response}")

Human: Hello! How are you today?
GPT: "thoughts🤔" The user greeted me; respond warmly and offer help.
"actions🚀" []
"value👉" Great working with you today!



In [7]:
image_path = "/home/jack/Projects/yixin-llm/images_train/23251064_F5.jpg"
question = "What type of diagnostic imaging does this image represent?"

image_conversation = []

response, image_conversation = chat_with_model(
    question, 
    image_paths=image_path, 
    conversation_history=image_conversation
)
print(f"Human: {question}\n")
print(f"GPT: {response}\n")


Human: What type of diagnostic imaging does this image represent?

GPT: "thoughts🤔" To accurately determine the modality, I will use a medical image classification tool.
"actions🚀" [{"API_name": "BiomedClip", "API_params": {}}]
"value👉" I'll use BiomedClip to complete this request based on our current analysis.




In [9]:
question2 = "Following up on the previous analysis, what is the main focus of the image?"

response2, image_conversation = chat_with_model(
    question2, 
    image_paths="/home/jack/Projects/yixin-llm/images_train/WORD_0049_0137.jpg",
    conversation_history=image_conversation
)
print(f"Human: {question2}\n")
print(f"GPT: {response2}")

Human: Following up on the previous analysis, what is the main focus of the image?

GPT: "thoughts🤔" I should utilize a classification model to identify the primary anatomical structures depicted in the image.
"actions🚀" [{"API_name": "BiomedClip", "API_params": {}}]
"value👉" I'll use BiomedClip to complete this request based on our current analysis.



In [11]:
question3 = "Now I have a second image. \nRegister this new scan with the previous one."

response3, image_conversation = chat_with_model(
    question3, 
    image_paths="/home/jack/Projects/yixin-llm/yixin-llm-data/UltraSam/dataset/BrEaST/BrEaST-Lesions_USG-images_and_masks-Dec-15-2023/images/BrEaST-Lesions_USG-images_and_masks-Dec-15-2023__00224.png",
    conversation_history=image_conversation
)
print(f"Human: {question3}\n")
print(f"GPT: {response3}")

Human: Now I have a second image. 
Register this new scan with the previous one.

GPT: "thoughts🤔" This is an image registration task. I need to use UniGradICON to align the moving image to the fixed image.
"actions🚀" [{"API_name": "UniGradICON", "API_params": {"modality": "CT"}}]
"value👉" I'll use UniGradICON to complete this request based on our current analysis.



In [12]:
print_conversation_history(image_conversation)

=== Conversation History ===
1. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/images_train/23251064_F5.jpg]
   What type of diagnostic imaging does this image represent?

2. GPT:
   "thoughts🤔" To accurately determine the modality, I will use a medical image classification tool.
"actions🚀" [{"API_name": "BiomedClip", "API_params": {}}]
"value👉" I'll use BiomedClip to complete this request based on our current analysis.


3. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/images_train/WORD_0049_0137.jpg]
   Following up on the previous analysis, what is the main focus of the image?

4. GPT:
   "thoughts🤔" I should utilize a classification model to identify the primary anatomical structures depicted in the image.
"actions🚀" [{"API_name": "BiomedClip", "API_params": {}}]
"value👉" I'll use BiomedClip to complete this request based on our current analysis.


5. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/yixin-llm-data/UltraSam/dataset/BrEaST/BrEaST-Lesions_USG-images_and_masks-Dec-15

In [10]:
reset_conversation()

[]

In [11]:
response, history = chat_with_model("hi")
print(f"Human: hi")
print(f"GPT: {response}")

Human: hi
GPT: "thoughts🤔" The user is greeting me. I should respond warmly and offer assistance.
"actions🚀" []
"value👉" Glad I could help!



In [17]:
image_path = ["/home/jack/Projects/yixin-llm/yixin-llm-data/instruct_dataset/mimic-cxr-5k/5k/263caa43-d7645fc2-ec153511-fefb77fe-01cf1b02.jpg", "/home/jack/.cache/kagglehub/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/versions/10/full-fundus/full-fundus/ORIGA-531.png"]
question = "Align this for image fusion."

image_conversation = []

response, image_conversation = chat_with_model(
    question, 
    image_paths=image_path, 
    conversation_history=image_conversation
)
print(f"Human: {question}\n")
print(f"GPT: {response}\n")

Human: Align this for image fusion.

GPT: "thoughts🤔" This is an image registration task. I need to use UniGradICON to align the moving image to the fixed image.
"actions🚀" [{"API_name": "UniGradICON", "API_params": {"modality": "CT"}}]
"value👉" I'll use UniGradICON to complete this request based on our current analysis.




In [ ]:
question2 = "Could you generate a structured report for the attached MRI scan?"

response2, image_conversation = chat_with_model(
    question2, 
    conversation_history=image_conversation
)
print(f"Human: {question2}\n")
print(f"GPT: {response2}")

Human: Could you generate a structured report for the attached MRI scan?

GPT: "thoughts🤔" To provide an accurate and comprehensive structured report, I will utilize the ChatCAD-G tool to analyze the MRI scan thoroughly.
"actions🚀" [{"API_name": "ChatCAD-G", "API_params": {}}]
"value👉" I'll use ChatCAD-G to complete this request based on our current analysis.



In [19]:
question3 = "Building on the results so far, segment the target anatomy inside box [171.0, 50.0, 298.0, 193.0] for ultrasound image."

response3, image_conversation = chat_with_model(
    question3, 
    conversation_history=image_conversation
)
print(f"Human: {question3}\n")
print(f"GPT: {response3}")


Human: Building on the results so far, segment the target anatomy inside box [171.0, 50.0, 298.0, 193.0] for ultrasound image.

GPT: "thoughts🤔" This is an ultrasound segmentation task; I'll call the UltraSAM tool with a bbox prompt.
"actions🚀" [{"API_name": "UltraSAM", "API_params": {"image": 222, "prompt": {"bboxes": [[171.0, 50.0, 298.0, 193.0]]}}}]
"value👉" I'll use UltraSAM to complete this request based on our current analysis.



In [21]:
print_conversation_history(image_conversation)

=== Conversation History ===
1. HUMAN:
   [Image: /home/jack/Projects/yixin-llm/yixin-llm-data/instruct_dataset/mimic-cxr-5k/5k/263caa43-d7645fc2-ec153511-fefb77fe-01cf1b02.jpg]
   [Image: /home/jack/.cache/kagglehub/datasets/deathtrooper/multichannel-glaucoma-benchmark-dataset/versions/10/full-fundus/full-fundus/ORIGA-531.png]
   Align this for image fusion.

2. GPT:
   "thoughts🤔" This is an image registration task. I need to use UniGradICON to align the moving image to the fixed image.
"actions🚀" [{"API_name": "UniGradICON", "API_params": {"modality": "CT"}}]
"value👉" I'll use UniGradICON to complete this request based on our current analysis.


3. HUMAN:
   Could you generate a structured report for the attached MRI scan?

4. GPT:
   "thoughts🤔" To provide an accurate and comprehensive structured report, I will utilize the ChatCAD-G tool to analyze the MRI scan thoroughly.
"actions🚀" [{"API_name": "ChatCAD-G", "API_params": {}}]
"value👉" I'll use ChatCAD-G to complete this request 

In [ ]:
# Interactive chat session
print("Starting interactive chat. Type 'quit' to exit, 'reset' to clear history.")
interactive_history = []

while True:
    user_input = input("\nHuman: ")
    
    if user_input.lower() == 'quit':
        break
    elif user_input.lower() == 'reset':
        interactive_history = []
        print("Conversation history cleared.")
        continue
    
    try:
        response, interactive_history = chat_with_model(user_input, conversation_history=interactive_history)
        print(f"GPT: {response}")
    except Exception as e:
        print(f"Error: {e}")

Starting interactive chat. Type 'quit' to exit, 'reset' to clear history.
